# MovieLens 32M — independent LightGBM ranking

Consume historical and full-Train Validation fusion artifacts. This notebook contains no retriever algorithms and never trains retriever snapshots. Generate those independently using `rank_snapshot` in notebooks 03–07 and fuse them in Notebook 08.


## 1. Environment and paths

Place this repository's readable `src/recsys/`, `configs/` and `tests/` directories in the Google Drive project directory below. No other notebook needs to have run in this runtime. The existing dataset and split artifacts must already exist for real experiments. The source path and file hashes are checked before imports; restart the runtime if a different or edited package is cached. Dependency requirements and installation instructions are in the project README; no automatic package installation runs here.


In [ ]:
from pathlib import Path
import sys
import importlib.util
RUN_MODE = "smoke_test"
if RUN_MODE == "smoke_test":
    import os
    for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[_thread_variable] = "1"  # bound synthetic CPU native-library concurrency before imports
SUPPORTED_MODES = ('smoke_test', 'full_train', 'load_evaluate')
if RUN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported mode {RUN_MODE!r}; choose {SUPPORTED_MODES}")
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "personal_projects/movielens_hybrid_recsys"
)
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
SRC_DIR = PROJECT_DIR / "src"
if not (SRC_DIR / "recsys" / "__init__.py").is_file():
    raise FileNotFoundError(f"Readable source missing: {SRC_DIR / 'recsys'}. Copy or clone the repository there.")
import hashlib
_source_hash = hashlib.sha256()
for _file in sorted((SRC_DIR / "recsys").rglob("*.py")):
    _source_hash.update(str(_file.relative_to(SRC_DIR)).encode())
    _source_hash.update(_file.read_bytes())
_loaded = sys.modules.get("recsys")
if _loaded is not None:
    if (not Path(_loaded.__file__).resolve().is_relative_to(SRC_DIR.resolve()) or
        getattr(_loaded, "__notebook_source_hash__", None) != _source_hash.hexdigest()):
        raise RuntimeError("A stale/different recsys module is cached. Restart this runtime before importing.")
sys.path = [str(SRC_DIR)] + [p for p in sys.path if p != str(SRC_DIR) and "recsys_source" not in p]
import recsys
recsys.__notebook_source_hash__ = _source_hash.hexdigest()
from recsys.runtime import environment
from recsys.smoke import run_smoke
print("Source:", recsys.__file__)
print(environment("/content" if Path("/content").exists() else "/tmp"))


In [ ]:
if RUN_MODE == "smoke_test":
    print(run_smoke(PROJECT_DIR, 'ranking'))


## 2. Configuration and dependencies

`full_train` here fits LambdaMART from historical candidates, preserving the external partitions. `load_evaluate` loads an existing ranker. Historical features must come from strictly earlier retrievers, and all label timestamps must remain inside external Train. No in-sample full-Train substitute is accepted.


In [ ]:
import json
from recsys.rankers.workflow import RankingExperiment
CONFIG = json.loads((PROJECT_DIR / "configs" / "lightgbm.json").read_text())
CONFIG.update(seed=42, cpu_threads=4, bootstrap=300)
TRAIN_FUSION = None  # historical rank_train fusion directory
VALIDATION_FUSION = None  # full-Train Validation fusion directory
RANKER_CHECKPOINT = None  # required only for load_evaluate
RANKER_OUTPUT = None


## 3. Verify frozen split, time and feature contracts

Check the split checksum, source/catalog/mapping provenance and fixed Validation population. Historical training cutoff must precede every ranking label; the label window must end before external Validation. Feature names are an explicit allow-list excluding labels, feedback kinds and future statistics.


In [ ]:
if RUN_MODE != "smoke_test":
    if VALIDATION_FUSION is None:
        raise ValueError("Set the saved full-Train Validation fusion path.")
    ranking = RankingExperiment(PROJECT_DIR, TRAIN_FUSION, VALIDATION_FUSION, CONFIG,
                                RUN_MODE, checkpoint=RANKER_CHECKPOINT)


## 4. Leakage-aware ranking examples and query groups

Keep retrieved observed positives and observed low ratings, then sample a configurable number of unobserved candidates per query. Unknown zero labels are weak supervision, not verified negatives. Count queries with no retrieved positive and exclude them only from fitting; unconditional evaluation retains them. Query-contiguous Parquet batches are written without materializing the full feature table. Record counts, candidate recall ceiling, boundaries, sample-selection limitations and snapshot/full-model feature-distribution differences.


In [ ]:
examples = None
if RUN_MODE != "smoke_test":
    examples = ranking.prepare_examples()
    if examples is not None:
        print(json.dumps(json.loads((examples / "sampling_report.json").read_text()), indent=2))


## 5. LambdaMART training or checkpoint load

The migrated LambdaMART objective uses LightGBM's native `Dataset`/`Sequence` interface to stream raw Parquet features in bounded row groups. LightGBM still needs RAM for its compressed bins, labels, gradients and histograms, checked before fitting. The sklearn `LGBMRanker` helper remains available for small in-memory experiments. Both optimize `lambdarank`; the streaming API avoids assembling the complete raw table. A fixed Validation panel controls early stopping; Test is unopened.


In [ ]:
if RUN_MODE != "smoke_test":
    RANKER_CHECKPOINT = ranking.fit_or_load(examples)
    print("Saved ranker:", RANKER_CHECKPOINT)


## 6. Validation: RRF versus LightGBM

Evaluate every fixed eligible-positive user. Report unconditional and pool-hit-conditional Recall/NDCG/HitRate, coverage, popularity concentration and candidate recall ceiling. Inference ranks the entire unsampled union. A ranker cannot recover positives absent from the pool.


In [ ]:
import pandas as pd
from IPython.display import display
if RUN_MODE != "smoke_test":
    RANKER_OUTPUT = ranking.evaluate()
    summary = json.loads((RANKER_OUTPUT / "metrics.json").read_text())
    display(pd.DataFrame([{**reports[0], "model": name} for name, reports in summary["metrics"].items()]))
    display(pd.DataFrame(summary["conditional_on_pool_hit"]["lightgbm"]))
    print("Validation result:", RANKER_OUTPUT)


## 7. Feature importance, resources and persistence

Gain/split importance is descriptive, not causal. Checkpoints contain the model, feature contract, source hashes, input-artifact references, sampling report, history, versions and measured resources. Complete matching stages are reused; no previous run is overwritten.


In [ ]:
if RUN_MODE != "smoke_test":
    importance = pd.read_parquet(RANKER_CHECKPOINT / "feature_importance.parquet")
    display(importance.sort_values("gain", ascending=False))
    print(json.loads((RANKER_CHECKPOINT / "training_history.json").read_text())["profile"])


## 8. Limitations and handoff

Sampling changes the label balance; dropping no-positive training queries creates selection bias. Earlier snapshot embeddings, score scales, catalogs and histories differ from full-Train inference. The saved feature-shift diagnostics also reflect time/population changes and do not establish unbiased transfer. Exposure bias and rating bursts remain observational limitations. No Test tuning or real training was performed during framework construction.

Pass the selected ranker checkpoint, its Validation result and the matching retrieval/fusion paths to Notebook 10 after reviewing Validation results. No real ranking result is prefilled here.
